In [ ]:
# necessary imports
import os 
import random 
import numpy as np
import pandas as pd
import polars as pl 
import pickle, gzip
import tensorflow as tf 
from keras.utils import to_categorical  
import data_preprocessing
import data_models

## Read data and assing age groups

In [ ]:
# Define schema override
schema = {
    "Age": pl.Utf8,
}

# Read CSV with schema
raw_data = pl.read_csv("Data/SDBII_5.csv", schema_overrides=schema)

raw_data = raw_data.with_columns(
    pl.col("ts").str.to_datetime()
)

raw_data_sorted = raw_data.sort(["PtID", "ts"])
raw_data_sorted = raw_data_sorted.unique(subset=["PtID", "ts"], keep="first")

In [ ]:
n_subjects = raw_data_sorted.select(pl.col("PtID").n_unique()).item()
print(n_subjects)

2499


In [ ]:
df_age = data_preprocessing.convert_age_strings(raw_data_sorted)
df_age = data_preprocessing.assign_age_group_from_string_ranges(df_age)

## extract subjects with more data as test subjects

In [5]:
counts = (
    df_age
    .group_by(["AgeGroup", "PtID"])
    .agg(pl.count("GlucoseCGM").alias("count"))
)

top5_per_age = (
    counts
    .sort(["AgeGroup", "count"], descending=[False, True])
    .with_columns(
        pl.col("count")
        .rank("dense", descending=True)
        .over("AgeGroup")
        .alias("rank")
    )
    .filter(pl.col("rank") <= 10)
)

print(top5_per_age)

shape: (40, 4)
┌──────────┬──────────────────────┬────────┬──────┐
│ AgeGroup ┆ PtID                 ┆ count  ┆ rank │
│ ---      ┆ ---                  ┆ ---    ┆ ---  │
│ i32      ┆ str                  ┆ u32    ┆ u32  │
╞══════════╪══════════════════════╪════════╪══════╡
│ 0        ┆ 73.0_PEDAP           ┆ 118667 ┆ 1    │
│ 0        ┆ 53.0_PEDAP           ┆ 110035 ┆ 2    │
│ 0        ┆ 38.0_PEDAP           ┆ 109002 ┆ 3    │
│ 0        ┆ 87.0_PEDAP           ┆ 108795 ┆ 4    │
│ 0        ┆ 28.0_SENCE           ┆ 108294 ┆ 5    │
│ …        ┆ …                    ┆ …      ┆ …    │
│ 3        ┆ 91.0_WISDM           ┆ 105035 ┆ 6    │
│ 3        ┆ 205.0_WISDM          ┆ 104679 ┆ 7    │
│ 3        ┆ 165.0_WISDM          ┆ 104096 ┆ 8    │
│ 3        ┆ LIB193307_T1DGranada ┆ 104042 ┆ 9    │
│ 3        ┆ 47.0_WISDM           ┆ 103645 ┆ 10   │
└──────────┴──────────────────────┴────────┴──────┘


In [6]:
# Convert top5_per_age into dictionary {AgeGroup: [PtIDs]}
result_dict = (
    top5_per_age
    .group_by("AgeGroup")
    .agg(pl.col("PtID").alias("PtIDs"))
    .to_dict(as_series=False)
)


# Optionally, make it cleaner: map AgeGroup → list of IDs
age_to_ids = dict(zip(result_dict["AgeGroup"], result_dict["PtIDs"]))

subjects = list(age_to_ids.values())
subjects = np.array(subjects)

In [9]:
children = df_age.filter(pl.col("PtID").is_in(subjects[0]) & (pl.col("AgeGroup") ==0))
teenagers = df_age.filter(pl.col("PtID").is_in(subjects[1])& (pl.col("AgeGroup") ==1))
adults = df_age.filter(pl.col("PtID").is_in(subjects[2])& (pl.col("AgeGroup") ==2))
seniors = df_age.filter(pl.col("PtID").is_in(subjects[3])& (pl.col("AgeGroup") ==3))

df_filtered_age = df_age.filter(~(pl.col("PtID").is_in(subjects[0])))
df_filtered_age = df_filtered_age.filter(~(pl.col("PtID").is_in(subjects[1])))
df_filtered_age = df_filtered_age.filter(~(pl.col("PtID").is_in(subjects[2])))
df_filtered_age = df_filtered_age.filter(~(pl.col("PtID").is_in(subjects[3])))

In [10]:
n_subjects_before = df_age.select(pl.col("PtID").n_unique()).item()
print(n_subjects_before)

2499


In [11]:
subjects_by_age_before = (
    df_age.group_by("AgeGroup")
      .agg(pl.col("PtID").n_unique().alias("n_subjects"))
)
print(subjects_by_age_before)

shape: (4, 2)
┌──────────┬────────────┐
│ AgeGroup ┆ n_subjects │
│ ---      ┆ ---        │
│ i32      ┆ u32        │
╞══════════╪════════════╡
│ 0        ┆ 380        │
│ 3        ┆ 1035       │
│ 1        ┆ 407        │
│ 2        ┆ 766        │
└──────────┴────────────┘


In [14]:
df_nonan= df_age.drop_nans(subset=["GlucoseCGM"])

In [16]:
summary = (
    df_nonan
    .group_by("AgeGroup")
    .agg([
        pl.col("PtID").n_unique().alias("n_patients"),
        pl.col("GlucoseCGM").mean().alias("avg_days"),
        pl.col("GlucoseCGM").std().alias("min_days"),
        pl.col("GlucoseCGM").count().alias("median_days"),
    ])
    .sort("AgeGroup")
)

print("\nSummary per age group:")
print(summary)


Summary per age group:
shape: (4, 5)
┌──────────┬────────────┬────────────┬───────────┬─────────────┐
│ AgeGroup ┆ n_patients ┆ avg_days   ┆ min_days  ┆ median_days │
│ ---      ┆ ---        ┆ ---        ┆ ---       ┆ ---         │
│ i32      ┆ u32        ┆ f64        ┆ f64       ┆ u32         │
╞══════════╪════════════╪════════════╪═══════════╪═════════════╡
│ 0        ┆ 380        ┆ 178.88646  ┆ 78.171729 ┆ 19837509    │
│ 1        ┆ 407        ┆ 185.190641 ┆ 82.842668 ┆ 16785305    │
│ 2        ┆ 766        ┆ 166.956574 ┆ 70.220832 ┆ 35568309    │
│ 3        ┆ 1035       ┆ 159.408897 ┆ 64.598251 ┆ 33484933    │
└──────────┴────────────┴────────────┴───────────┴─────────────┘


In [12]:
n_subjects = df_filtered_age.select(pl.col("PtID").n_unique()).item()
print(n_subjects)

2459


In [13]:
subjects_by_age = (
    df_filtered_age.group_by("AgeGroup")
      .agg(pl.col("PtID").n_unique().alias("n_subjects"))
)
print(subjects_by_age)

shape: (4, 2)
┌──────────┬────────────┐
│ AgeGroup ┆ n_subjects │
│ ---      ┆ ---        │
│ i32      ┆ u32        │
╞══════════╪════════════╡
│ 3        ┆ 1025       │
│ 0        ┆ 370        │
│ 1        ┆ 395        │
│ 2        ┆ 754        │
└──────────┴────────────┘


In [14]:
avg_per_age = (
    df_filtered_age.group_by(["AgeGroup", "PtID"])
      .agg(pl.count("GlucoseCGM").alias("n_datapoints"))
      .group_by("AgeGroup")
      .agg(pl.col("n_datapoints").mean().alias("avg_CGM_points_per_age_group"))
)

print(avg_per_age)

shape: (4, 2)
┌──────────┬──────────────────────────────┐
│ AgeGroup ┆ avg_CGM_points_per_age_group │
│ ---      ┆ ---                          │
│ i32      ┆ f64                          │
╞══════════╪══════════════════════════════╡
│ 1        ┆ 39418.772152                 │
│ 2        ┆ 42075.916446                 │
│ 0        ┆ 50681.281081                 │
│ 3        ┆ 31605.37561                  │
└──────────┴──────────────────────────────┘


In [15]:
avg_per_age_filt = (
    df_age.group_by(["AgeGroup", "PtID"])
      .agg(pl.count("GlucoseCGM").alias("n_datapoints"))
      .group_by("AgeGroup")
      .agg(pl.col("n_datapoints").mean().alias("avg_CGM_points_per_age_group"))
)

print(avg_per_age_filt)

shape: (4, 2)
┌──────────┬──────────────────────────────┐
│ AgeGroup ┆ avg_CGM_points_per_age_group │
│ ---      ┆ ---                          │
│ i32      ┆ f64                          │
╞══════════╪══════════════════════════════╡
│ 3        ┆ 32352.592271                 │
│ 0        ┆ 52203.971053                 │
│ 2        ┆ 46433.82376                  │
│ 1        ┆ 41241.535627                 │
└──────────┴──────────────────────────────┘


In [ ]:
# Group by "age" and count total values  
result = (
    df_age.group_by("AgeGroup")
      .agg(pl.count("GlucoseCGM").alias("total_count"))
)

print(result)

shape: (4, 2)
┌──────────┬─────────────┐
│ AgeGroup ┆ total_count │
│ ---      ┆ ---         │
│ i32      ┆ u32         │
╞══════════╪═════════════╡
│ 0        ┆ 19837509    │
│ 3        ┆ 33484933    │
│ 1        ┆ 16785305    │
│ 2        ┆ 35568309    │
└──────────┴─────────────┘


In [ ]:
# Group by "age" and count total values  
result = (
    df_filtered_age.group_by("AgeGroup")
      .agg(pl.count("GlucoseCGM").alias("total_count"))
)

print(result)

shape: (4, 2)
┌──────────┬─────────────┐
│ AgeGroup ┆ total_count │
│ ---      ┆ ---         │
│ i32      ┆ u32         │
╞══════════╪═════════════╡
│ 3        ┆ 32395510    │
│ 0        ┆ 18752074    │
│ 1        ┆ 15570415    │
│ 2        ┆ 31725241    │
└──────────┴─────────────┘


In [ ]:

count = raw_data_sorted.select([
    pl.col(c).is_not_null().sum().alias(c)
    for c in raw_data_sorted.columns
])
count

ts,PtID,GlucoseCGM,Age,Sex,Database
u32,u32,u32,u32,u32,u32
237842393,237842393,105676056,237842393,237842393,237842393


In [ ]:
# Count values below 71
count = raw_data_sorted.select(
    pl.col("GlucoseCGM").filter(pl.col("GlucoseCGM") < 71).count()
).item()
count

4032032

In [ ]:

df_clean = (
    df_age.drop_nulls()                          
)

records_per_person = df_clean.group_by("PtID").len().rename({"len": "num_records"})

# Compute days per person (num_records / 288)
days_per_person = records_per_person.with_columns(
    (pl.col("num_records") / 288).alias("days_of_data")
)

# Compute average across all persons
avg_days = days_per_person.select(pl.col("days_of_data").mean()).item()

print(days_per_person)
print(f"\nAverage days of data per person: {avg_days:.2f}")
print(f"\nmin days of data per person: {days_per_person.min()}")
print(f"\nmean days of data per person: {days_per_person.mean()}")
print(f"\nmax days of data per person: {days_per_person.max()}")

shape: (2_499, 3)
┌──────────────────────┬─────────────┬──────────────┐
│ PtID                 ┆ num_records ┆ days_of_data │
│ ---                  ┆ ---         ┆ ---          │
│ str                  ┆ u32         ┆ f64          │
╞══════════════════════╪═════════════╪══════════════╡
│ 89.0_SENCE           ┆ 52562       ┆ 182.506944   │
│ 56.0_RT-CGM          ┆ 30694       ┆ 106.576389   │
│ 93.0_RBG             ┆ 64988       ┆ 225.652778   │
│ LIB193367_T1DGranada ┆ 75267       ┆ 261.34375    │
│ LIB193980_T1DGranada ┆ 11423       ┆ 39.663194    │
│ …                    ┆ …           ┆ …            │
│ 46.0_RBG             ┆ 70694       ┆ 245.465278   │
│ LIB193722_T1DGranada ┆ 46287       ┆ 160.71875    │
│ 60.0_CITY            ┆ 50839       ┆ 176.524306   │
│ 98.0_WISDM           ┆ 54846       ┆ 190.4375     │
│ 47.0_CITY            ┆ 39653       ┆ 137.684028   │
└──────────────────────┴─────────────┴──────────────┘

Average days of data per person: 146.83

min days of data per p

In [ ]:
df_clean = df_age.drop_nulls()

days_per_person = (
    df_clean
    .group_by(["AgeGroup", "PtID"])
    .len()
    .rename({"len": "num_records"})
    .with_columns((pl.col("num_records") / 288).alias("days_of_data"))
)

print(days_per_person)

summary = (
    days_per_person
    .group_by("AgeGroup")
    .agg([
        pl.col("PtID").n_unique().alias("n_patients"),
        pl.col("days_of_data").mean().alias("avg_days"),
        pl.col("days_of_data").min().alias("min_days"),
        pl.col("days_of_data").median().alias("median_days"),
        pl.col("days_of_data").max().alias("max_days"),
    ])
    .sort("AgeGroup")
)

print("\nSummary per age group:")
print(summary)

shape: (2_588, 4)
┌──────────┬──────────────────────┬─────────────┬──────────────┐
│ AgeGroup ┆ PtID                 ┆ num_records ┆ days_of_data │
│ ---      ┆ ---                  ┆ ---         ┆ ---          │
│ i32      ┆ str                  ┆ u32         ┆ f64          │
╞══════════╪══════════════════════╪═════════════╪══════════════╡
│ 3        ┆ 131.0_SHD            ┆ 1929        ┆ 6.697917     │
│ 2        ┆ LIB193827_T1DGranada ┆ 6101        ┆ 21.184028    │
│ 2        ┆ LIB193956_T1DGranada ┆ 7310        ┆ 25.381944    │
│ 1        ┆ 76.0_DLCP3           ┆ 48904       ┆ 169.805556   │
│ 2        ┆ 40.0_RBG             ┆ 62277       ┆ 216.239583   │
│ …        ┆ …                    ┆ …           ┆ …            │
│ 2        ┆ 177.0_RT-CGM         ┆ 47765       ┆ 165.850694   │
│ 2        ┆ 424.0_RT-CGM         ┆ 9691        ┆ 33.649306    │
│ 0        ┆ 84.0_RT-CGM          ┆ 19873       ┆ 69.003472    │
│ 1        ┆ 470.0_RT-CGM         ┆ 1724        ┆ 5.986111     │
│ 0    

## Preprocessing

In [ ]:
# removes outliers
cleaned_static = data_preprocessing.remove_outliers_polars(df_filtered_age, value="GlucoseCGM")

data_cleanedpd = cleaned_static.to_pandas()
# values need to be sorted before imputation
data_cleanedpd = data_cleanedpd.sort_values(by=["PtID", "ts"]).reset_index(drop=True)

# linear interpolation is used for gaps which are less than 30 minute (6 consecuitve datapoints)
data_cleanedpd["GlucoseCGM"] = data_cleanedpd.groupby("PtID")["GlucoseCGM"].transform(
    lambda x: data_preprocessing.gap_limited_interpolation(x, limit=6)
)

# stineman interpolation for values between 30 adn 120 minutes
df_interpolated_stine = data_cleanedpd.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.interpolate_stineman_group(x, timestamp = "ts", value = "GlucoseCGM", llimit=6, ulimit=24))
df_interpolated_stine = df_interpolated_stine.sort_index()

df_interpolated_stine = df_interpolated_stine[["ts", "GlucoseCGM", "PtID",	"AgeGroup"]]

## class assgnment

In [ ]:
classes_df = df_interpolated_stine.sort_values(by=["PtID", "ts"]).reset_index(drop=True)
                                                    
# initially all classes are assigned a class -1
classes_df["Class1"] = -1

# hypoglycemic values are defined as class 0 with values equal and lower than 70 mg/dL
classes_df.loc[classes_df["GlucoseCGM"] <= 70, "Class1"] = 0

# calls the function class_generation to assing classes based on wanted intervals before hypogylcemia for each subject separately
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.class_generation(x,"ts", 5, 20, 1, col_name = "Class1")) # 5-15 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.class_generation(x,"ts", 20, 50, 2, col_name = "Class1")) # 15-30 min
classes_df = classes_df.groupby("PtID", group_keys=False).apply(lambda x: data_preprocessing.class_generation(x,"ts", 50, 125, 3, col_name = "Class1"))  # 30-60 min
classes_df.loc[classes_df["Class1"] == -1, "Class1"] = 4

# sorts the glucose values by patient id and timestamps
df_sorted = classes_df.sort_values(by=["PtID", "ts"])
# reindexes based on the sorted dataset
df_sorted = df_sorted.reset_index(drop=True)
print(df_sorted["Class1"].value_counts())

Class1
4    209063256
3      7108303
0      5932994
2      3195689
1      1741703
Name: count, dtype: int64


In [24]:
df_sorted.groupby("AgeGroup")["Class1"].value_counts()

AgeGroup  Class1
0         4         31129113
          3          1145134
          0           610954
          2           518626
          1           286702
1         4         38707350
          3           964866
          0           864349
          2           436475
          1           241667
2         4         69000145
          3          2614032
          0          2375879
          2          1172978
          1           634334
3         4         70226648
          3          2384271
          0          2081812
          2          1067610
          1           579000
Name: count, dtype: int64

## features

In [25]:
print(df_sorted["GlucoseCGM"].min())
print(df_sorted["GlucoseCGM"].max())
df_scaled = data_preprocessing.normalize_data(df_sorted, "GlucoseCGM")
print(df_scaled["PtID"].nunique())
df_scaled = {age: group for age, group in df_scaled.groupby("AgeGroup")}

40.0
500.0
2459


## time series generation

In [ ]:
results = {}

for age, group_df in df_scaled.items():
    print(f"Processing age group {age}...")
    X_data, Y_data = data_preprocessing.create_X_Y(group_df, feature="PtID", labels = "Class1", sample_count=25, hours=2, modus="h")
    results[age] = (X_data, Y_data)

Processing age group 0...
Processing age group 1...
Processing age group 2...
Processing age group 3...


In [ ]:
# Save (compressed)
with gzip.open("train_data_120min.pkl.gz", "wb") as f:
    pickle.dump(results, f, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# Load data
with gzip.open("Data/train_data.pkl.gz", "rb") as f:
    results = pickle.load(f)

## group based models

In [ ]:
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)

X_data_train = []
Y_data_train = []

for age, (X_data, Y_data) in results.items():
    print(f"Splitting data for age group {age}...")
    
    # assuming X_data and Y_data are lists (or arrays) where each entry corresponds to one subject
    subject_indices = list(range(len(X_data)))

    X_data_train.append(X_data) 
    Y_data_train.append(Y_data)

Splitting data for age group 0...
Splitting data for age group 1...
Splitting data for age group 2...
Splitting data for age group 3...


In [5]:
name = ["Children" , "Teenagers" , "Adults", "Seniors"]
for i in range(0,4):
    print(i)
    print(name[i])
    print(len(X_data_train[i]))

0
Children
370
1
Teenagers
393
2
Adults
752
3
Seniors
1015


In [ ]:
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)


X_train_fold = data_preprocessing.flatten_data(X_data_train[0], modus = "input", shape_f = 25, dim = 1)
Y_train_fold = data_preprocessing.flatten_data(Y_data_train[0], modus = "output", shape_f = 25, dim = 1)

Y_train_encoded = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype("int32")

BUFFER_SIZE = min(1000000, int(len(X_train_fold)))
BATCH_SIZE = 256

# reshapes Y_train, val, and test
Y_train_rs = Y_train_fold.reshape(-1).astype("int32")

# converts train, val, and test data into tensors
X_train_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_train_t = tf.convert_to_tensor(Y_train_encoded, dtype=tf.int32)

# shuffles train data and applies batch size 
train_ds_t = tf.data.Dataset.from_tensor_slices((X_train_t, Y_train_t))
train_ds = train_ds_t.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4)
class_counts = np.bincount(Y_train_weights, minlength=4).astype(np.float32)
inv_freq = 1.0 / np.maximum(class_counts, 1)
alpha = inv_freq / np.mean(inv_freq)  # normalize mean≈1
print("Alpha weights:", alpha)

# Define focal loss with tuned alpha and gamma
focloss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5,          # focus more on hard examples
    alpha=alpha,        # per-class weights
    )


# builds and trains the FCN model
model_fcn = data_models.build_FCN((X_train_fold.shape[1], X_train_fold.shape[2]), 4) 
model_fcn.compile(loss=focloss, optimizer=optimizer, metrics=["categorical_accuracy"])
model_fcn.fit(train_ds, epochs=100, verbose=0)

name_fcn = "Models/Children.hdf5"
model_fcn.save(name_fcn)                                                                                                                                                                                                                                                                                                                                                                                                                                   

2026-03-12 22:03:06.662559: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-12 22:03:06.662583: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


(2380902, 25, 1)
(2380902, 1)
Alpha weights: [0.8199664  1.7591739  0.9751476  0.44571236]


2026-03-12 22:03:07.763162: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


(2276691, 25, 1)
(2276691, 1)
Alpha weights: [0.52087533 1.9194262  1.0690062  0.4906922 ]


2026-03-13 02:02:26.922363: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


In [ ]:
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)


X_train_fold = data_preprocessing.flatten_data(X_data_train[1], modus = "input", shape_f = 25, dim = 1)
Y_train_fold = data_preprocessing.flatten_data(Y_data_train[1], modus = "output", shape_f = 25, dim = 1)

Y_train_encoded = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype("int32")

BUFFER_SIZE = min(1000000, int(len(X_train_fold)))
BATCH_SIZE = 256

# reshapes Y_train, val, and test
Y_train_rs = Y_train_fold.reshape(-1).astype("int32")

# converts train, val, and test data into tensors
X_train_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_train_t = tf.convert_to_tensor(Y_train_encoded, dtype=tf.int32)

# shuffles train data and applies batch size 
train_ds_t = tf.data.Dataset.from_tensor_slices((X_train_t, Y_train_t))
train_ds = train_ds_t.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4)
class_counts = np.bincount(Y_train_weights, minlength=4).astype(np.float32)
inv_freq = 1.0 / np.maximum(class_counts, 1)
alpha = inv_freq / np.mean(inv_freq)  # normalize mean≈1
print("Alpha weights:", alpha)

# Define focal loss with tuned alpha and gamma
focloss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5,          # focus more on hard examples
    alpha=alpha,        # per-class weights
    )


# builds and trains the FCN model
model_fcn = data_models.build_FCN((X_train_fold.shape[1], X_train_fold.shape[2]), 4) 
model_fcn.compile(loss=focloss, optimizer=optimizer, metrics=["categorical_accuracy"])
model_fcn.fit(train_ds, epochs=100, verbose=0)

name_fcn = "Models/Teenagers.hdf5"
model_fcn.save(name_fcn)                                                                                                                                                                                                                                                                                                                                                                                                                                   

2026-03-14 23:26:47.953928: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Max
2026-03-14 23:26:47.954006: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-03-14 23:26:47.954013: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-03-14 23:26:47.954079: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-14 23:26:47.954104: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


(2276691, 25, 1)
(2276691, 1)


2026-03-14 23:26:48.499825: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-14 23:26:48.499851: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Alpha weights: [0.52087533 1.9194262  1.0690062  0.4906922 ]


2026-03-14 23:26:49.732953: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
## ohne klasse 4 mit weigths 
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)


X_train_fold = data_preprocessing.flatten_data(X_data_train[2], modus = "input", shape_f = 25, dim = 1)
Y_train_fold = data_preprocessing.flatten_data(Y_data_train[2], modus = "output", shape_f = 25, dim = 1)

Y_train_encoded = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype("int32")

BUFFER_SIZE = min(1000000, int(len(X_train_fold)))
BATCH_SIZE = 256

# reshapes Y_train, val, and test
Y_train_rs = Y_train_fold.reshape(-1).astype("int32")

# converts train, val, and test data into tensors
X_train_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_train_t = tf.convert_to_tensor(Y_train_encoded, dtype=tf.int32)

# shuffles train data and applies batch size 
train_ds_t = tf.data.Dataset.from_tensor_slices((X_train_t, Y_train_t))
train_ds = train_ds_t.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4)
class_counts = np.bincount(Y_train_weights, minlength=4).astype(np.float32)
inv_freq = 1.0 / np.maximum(class_counts, 1)
alpha = inv_freq / np.mean(inv_freq)  # normalize mean≈1
print("Alpha weights:", alpha)

# Define focal loss with tuned alpha and gamma
focloss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5,          # focus more on hard examples
    alpha=alpha,        # per-class weights
    )


# builds and trains the FCN model
model_fcn = data_models.build_FCN((X_train_fold.shape[1], X_train_fold.shape[2]), 4) 
model_fcn.compile(loss=focloss, optimizer=optimizer, metrics=["categorical_accuracy"])
model_fcn.fit(train_ds, epochs=100, verbose=0)

name_fcn = "Models/Adults.hdf5"
model_fcn.save(name_fcn)                                                                                                                                                                                                                                                                                                                                                                                                                                   

2026-03-13 22:55:10.017545: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-13 22:55:10.017587: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


(6365318, 25, 1)
(6365318, 1)
Alpha weights: [0.51101387 1.9505793  1.0586015  0.47980526]


2026-03-13 22:55:12.686339: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


(5786305, 25, 1)
(5786305, 1)
Alpha weights: [0.52994287 1.9390709  1.0547019  0.47628433]


2026-03-14 09:51:54.035373: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


In [ ]:
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)


X_train_fold = data_preprocessing.flatten_data(X_data_train[3], modus = "input", shape_f = 25, dim = 1)
Y_train_fold = data_preprocessing.flatten_data(Y_data_train[3], modus = "output", shape_f = 25, dim = 1)

Y_train_encoded = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype("int32")

BUFFER_SIZE = min(1000000, int(len(X_train_fold)))
BATCH_SIZE = 256

# reshapes Y_train, val, and test
Y_train_rs = Y_train_fold.reshape(-1).astype("int32")

# converts train, val, and test data into tensors
X_train_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_train_t = tf.convert_to_tensor(Y_train_encoded, dtype=tf.int32)

# shuffles train data and applies batch size 
train_ds_t = tf.data.Dataset.from_tensor_slices((X_train_t, Y_train_t))
train_ds = train_ds_t.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4)
class_counts = np.bincount(Y_train_weights, minlength=4).astype(np.float32)
inv_freq = 1.0 / np.maximum(class_counts, 1)
alpha = inv_freq / np.mean(inv_freq)  # normalize mean≈1
print("Alpha weights:", alpha)

# Define focal loss with tuned alpha and gamma
focloss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5,          # focus more on hard examples
    alpha=alpha,        # per-class weights
    )


# builds and trains the FCN model
model_fcn = data_models.build_FCN((X_train_fold.shape[1], X_train_fold.shape[2]), 4) 
model_fcn.compile(loss=focloss, optimizer=optimizer, metrics=["categorical_accuracy"])
model_fcn.fit(train_ds, epochs=100, verbose=0)

name_fcn = "Models/Seniors.hdf5"
model_fcn.save(name_fcn)                                                                                                                                                                                                                                                                                                                                                                                                                                   

2026-03-15 03:26:40.204591: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-15 03:26:40.204624: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


(5786305, 25, 1)
(5786305, 1)
Alpha weights: [0.52994287 1.9390709  1.0547019  0.47628433]


2026-03-15 03:26:44.704718: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


## population based

In [ ]:
# Load data
with gzip.open("Data/train_data.pkl.gz", "rb") as f:
    data = pickle.load(f)

In [ ]:
from sklearn.model_selection import train_test_split
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)

X_data_train = []
Y_data_train = []

for age, (X_data, Y_data) in results.items():
    print(f"Splitting data for age group {age}...")
    
    # assuming X_data and Y_data are lists (or arrays) where each entry corresponds to one subject
    subject_indices = list(range(len(X_data)))
 

    X_data_train.append(X_data) 
    Y_data_train.append(Y_data)

Splitting data for age group 0...
Splitting data for age group 1...
Splitting data for age group 2...
Splitting data for age group 3...


In [14]:
all_subjects_X = []
all_subjects_Y = []

for group in X_data_train:        # each age group
    for subjects in group:    # list of subjects in that group
        for subj in subjects:
            # subj is a list of arrays → combine into one array per subject
            subj_concat = np.concatenate(subj, axis=0)
            flattened_data = subj_concat.reshape(-1,25,1)
            all_subjects_X.append(flattened_data)

print(f"Total subjects: {len(all_subjects_X)}")
print(f"Shape of first subject data: {all_subjects_X[0].shape}")


for group in Y_data_train:        # each age group
    for subjects in group:    # list of subjects in that group
        for subj in subjects:
            # subj is a list of arrays → combine into one array per subject
            subj_concat = np.concatenate(subj, axis=0)
            all_subjects_Y.append(subj_concat)

print(f"Total subjects: {len(all_subjects_Y)}")
print(f"Shape of first subject data: {all_subjects_Y[0].shape}")

Total subjects: 2530
Shape of first subject data: (15351, 25, 1)
Total subjects: 2530
Shape of first subject data: (15351,)


In [ ]:
# ensures reproducability
seed_value= 42  
os.environ["PYTHONHASHSEED"]=str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.compat.v1.set_random_seed(seed_value)
session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1, inter_op_parallelism_threads=1)
sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
tf.compat.v1.keras.backend.set_session(sess)


X_train_fold = np.concatenate(all_subjects_X, axis=0)
Y_train_fold = np.concatenate(all_subjects_Y, axis=0)

Y_train_encoded = to_categorical(Y_train_fold)

# for class weights 
Y_train_weights = Y_train_fold.reshape(-1).astype("int32")

BUFFER_SIZE = min(1000000, int(len(X_train_fold)))
BATCH_SIZE = 512                                                                                                                                                                                                                                                                                                                                                                                                                                 

# reshapes Y_train, val, and test
Y_train_rs = Y_train_fold.reshape(-1).astype("int32")

# converts train, val, and test data into tensors
X_train_t = tf.convert_to_tensor(X_train_fold, dtype=tf.float32)
Y_train_t = tf.convert_to_tensor(Y_train_encoded, dtype=tf.int32)

# shuffles train data and applies batch size 
train_ds_t = tf.data.Dataset.from_tensor_slices((X_train_t, Y_train_t))
train_ds = train_ds_t.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4)
class_counts = np.bincount(Y_train_weights, minlength=4).astype(np.float32)
inv_freq = 1.0 / np.maximum(class_counts, 1)
alpha = inv_freq / np.mean(inv_freq)  # normalize mean≈1
print("Alpha weights:", alpha)

# Define focal loss with tuned alpha and gamma
focloss = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=1.5,          # focus more on hard examples
    alpha=alpha,        # per-class weights
    )


# builds and trains the FCN model
model_fcn = data_models.build_FCN((X_train_fold.shape[1], X_train_fold.shape[2]), 4) 
model_fcn.compile(loss=focloss, optimizer=optimizer, metrics=["categorical_accuracy"])
model_fcn.fit(train_ds, epochs=100,verbose=0)

name_fcn= "Models/Population.hdf5"
model_fcn.save(name_fcn)

2026-03-07 04:54:47.950590: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-07 04:54:47.950619: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Alpha weights: [0.5531253  1.9200532  1.0500472  0.47677448]


2026-03-07 04:54:57.003698: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
